In [ ]:
Goal = [
    [1, 2, 3],
    [8, 0, 4],
    [7, 6, 5]
]

Goal_dict = {}

for i in range(3):
    for j in range(3):
        Goal_dict[Goal[i][j]] = (i, j)

print("Goal:", Goal_dict)


Start = [
    [1, 2, 4],
    [3, 5, 7],
    [6, 8, 0]
]

Start_dict = {}

for i in range(3):
    for j in range(3):
        Start_dict[Start[i][j]] = (i, j)

print("Start:", Start_dict)


def is_valid(coordinate):
    return 0 <= coordinate[0] < 3 and 0 <= coordinate[1] < 3


def calculate(state, Goal_dict):
    hn = 0

    for tile, coordinate in state.items():

     
        if tile == 0:
            continue

        goal_coordinate = Goal_dict[tile]

        hn += (
            abs(coordinate[0] - goal_coordinate[0])
            + abs(coordinate[1] - goal_coordinate[1])
        )

    return hn


Action = {
    "up": [-1, 0],
    "down": [1, 0],
    "right": [0, 1],
    "left": [0, -1]
}


current = Start_dict
current_h = calculate(current, Goal_dict)

while current != Goal_dict:

    best_state = None
    best_h = float('inf')

    blank = current[0]

    for action, move in Action.items():

        new_blank = (
            blank[0] + move[0],
            blank[1] + move[1]
        )

        if not is_valid(new_blank):
            continue

       
        moving_tile = None

        for tile, coordinate in current.items():
            if coordinate == new_blank:
                moving_tile = tile
                break

        temp = current.copy()

      
        temp[moving_tile] = blank

        
        temp[0] = new_blank

  
        hn = calculate(temp, Goal_dict)

        if hn < best_h:
            best_h = hn
            best_state = temp

    if best_h >= current_h:
        print("Stuck at local minimum.")
        break

    current = best_state
    current_h = best_h

    print("h(n) =", current_h)
    print(current)


print("\nFinal state:")
print(current)

Goal: {1: (0, 0), 2: (0, 1), 3: (0, 2), 8: (1, 0), 0: (1, 1), 4: (1, 2), 7: (2, 0), 6: (2, 1), 5: (2, 2)}
Start: {1: (0, 0), 2: (0, 1), 4: (0, 2), 3: (1, 0), 5: (1, 1), 7: (1, 2), 6: (2, 0), 8: (2, 1), 0: (2, 2)}
h(n) = 11
{1: (0, 0), 2: (0, 1), 4: (0, 2), 3: (1, 0), 5: (1, 1), 7: (2, 2), 6: (2, 0), 8: (2, 1), 0: (1, 2)}
h(n) = 10
{1: (0, 0), 2: (0, 1), 4: (1, 2), 3: (1, 0), 5: (1, 1), 7: (2, 2), 6: (2, 0), 8: (2, 1), 0: (0, 2)}
Stuck at local minimum.

Final state:
{1: (0, 0), 2: (0, 1), 4: (1, 2), 3: (1, 0), 5: (1, 1), 7: (2, 2), 6: (2, 0), 8: (2, 1), 0: (0, 2)}


In [ ]:

import heapq

Goal = [
    [1, 2, 3],
    [8, 0, 4],
    [7, 6, 5]
]

Start = [
    [1, 2, 4],
    [3, 5, 7],
    [6, 8, 0]
]

# Convert a puzzle into a dictionary: tile -> (row, column)
def create_dict(board):
    state = {}

    for i in range(3):
        for j in range(3):
            state[board[i][j]] = (i, j)

    return state


Goal_dict = create_dict(Goal)
Start_dict = create_dict(Start)


# Check whether a coordinate is inside the board
def is_valid(coordinate):
    r, c = coordinate
    return 0 <= r < 3 and 0 <= c < 3


# Manhattan Distance heuristic h(n)
def calculate(state):
    hn = 0

    for tile, coordinate in state.items():
        if tile == 0:
            continue

        goal_coordinate = Goal_dict[tile]

        hn += (
            abs(coordinate[0] - goal_coordinate[0])
            + abs(coordinate[1] - goal_coordinate[1])
        )

    return hn


# Possible movements of the blank tile
Action = {
    "up": (-1, 0),
    "down": (1, 0),
    "right": (0, 1),
    "left": (0, -1)
}


# Convert dictionary state into a hashable tuple
def state_key(state):
    return tuple(state[tile] for tile in range(9))


# A* Search Algorithm
def astar(start_state):
    start = state_key(start_state)
    goal = state_key(Goal_dict)

    # Priority queue: (f(n), g(n), state)
    pq = [(calculate(start_state), 0, start)]

    cost = {start: 0}
    parent = {start: None}
    states = {start: start_state.copy()}

    while pq:
        f, g, current_key = heapq.heappop(pq)

        # Ignore outdated queue entries
        if g != cost[current_key]:
            continue

        current = states[current_key]

        # Goal reached
        if current_key == goal:
            path = []
            key = current_key

            while key is not None:
                path.append(states[key])
                key = parent[key]

            return path[::-1]

        # Find blank tile position
        blank = current[0]

        # Generate successor states
        for action, move in Action.items():
            new_blank = (
                blank[0] + move[0],
                blank[1] + move[1]
            )

            if not is_valid(new_blank):
                continue

            # Find tile adjacent to the blank
            moving_tile = None

            for tile, coordinate in current.items():
                if coordinate == new_blank:
                    moving_tile = tile
                    break

            # Create successor state
            temp = current.copy()
            temp[0] = new_blank
            temp[moving_tile] = blank

            next_key = state_key(temp)
            new_g = g + 1

            # Update if a shorter path is found
            if next_key not in cost or new_g < cost[next_key]:
                cost[next_key] = new_g
                parent[next_key] = current_key
                states[next_key] = temp

                h = calculate(temp)
                fn = new_g + h

                heapq.heappush(pq, (fn, new_g, next_key))

    return None


# Display solution path
print("Initial State:")
for row in Start:
    print(row)

print("\nInitial heuristic h(n):", calculate(Start_dict))

path = astar(Start_dict)

if path:
    print("\nSolution found!")
    print("Total moves:", len(path) - 1)

    for step, state in enumerate(path):
        print(f"\nStep {step}:")

        board = [[0] * 3 for _ in range(3)]

        for tile, (r, c) in state.items():
            board[r][c] = tile

        for row in board:
            print(row)

        print("h(n):", calculate(state))
else:
    print("No solution exists.")
